# SHAPER — full staged pipeline

**SHAP**ley-guided **PER**turbation learning: cooperative attribution of training-time
augmentation views in contrastive sequential recommendation.

This notebook calls the package/script functions of the repository — it contains **no
scientific logic of its own**. It is restartable and resume-aware: every stage is
idempotent (checkpoint manifest + value-table cache) and refuses confirmatory work
before the pilot-informed archive exists.

The first code cell makes the repository root importable, so the notebook works no
matter where Jupyter's working directory happens to be.

The registered protocol lives in:
- `specs/SHAPER_Implementation_Spec.md` (implementation source of truth)
- `specs/SHAPER_Paper_Structure.md` (paper/interpretation source of truth)

**Staged order (enforced by gates).** Sections must run in order: 5 data → 7 recipe →
8 pilots → 9 amendment + archive → then the confirmatory stages. If a stage is run out
of order it prints a `GATE:` message (e.g. `ARCHIVE_FREEZE` requires the recipe
calibration of the SAME run and the completed pilot seeds) instead of crashing.

**Safety:** the notebook never launches the full study from one cell. Run the sections
one at a time or execute cells explicitly.


## 1. Environment

In [1]:
# Make the repository root importable from ANY working directory
# (Jupyter does not put the repo on sys.path automatically).
import os, pathlib, sys

def _is_repo_root(d: pathlib.Path) -> bool:
    try:
        return (d / 'shaper').is_dir() and (d / 'scripts').is_dir()
    except (PermissionError, OSError):
        return False

def _subdirs_of(d: pathlib.Path):
    try:
        return [p for p in d.iterdir() if p.is_dir()]
    except (PermissionError, OSError):
        return []

REPO_ROOT = None
_candidates = []
try:  # package already importable -> derive the root from its location
    import shaper as _shaper
    _candidates.append(pathlib.Path(_shaper.__file__).resolve().parent.parent)
except Exception:
    pass
_cwd = pathlib.Path.cwd().resolve()
_candidates += [_cwd, *_cwd.parents]                        # cwd and all ancestors
for _p in (_cwd, *_cwd.parents[:2]):                         # + immediate subdirectories
    _candidates += _subdirs_of(_p)
for _cand in _candidates:
    if _is_repo_root(_cand):
        REPO_ROOT = str(_cand)
        break
if REPO_ROOT is None:
    raise RuntimeError('could not locate the SHAPER repository root (a directory containing '
                       'shaper/ and scripts/). Run this notebook from inside the repository.')
for _p in (REPO_ROOT, os.path.join(REPO_ROOT, 'scripts')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print('REPO_ROOT:', REPO_ROOT)

from scripts import run_all
from shaper.config import load_run_config
from shaper.provenance import environment_record
from shaper.schedules import configure_determinism

import json
env = environment_record(configure_determinism(), cwd=REPO_ROOT)
print(json.dumps({k: env[k] for k in ("python", "platform", "cpu_count", "torch_version", "cuda_available", "commit", "dirty")}, indent=2))

REPO_ROOT: /Users/mlouhichi/Desktop/Shapper
{
  "python": "3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]",
  "platform": "macOS-26.6-arm64-arm-64bit",
  "cpu_count": 12,
  "torch_version": "2.13.0",
  "cuda_available": false,
  "commit": "df3c89876feb4a747a6824359734b1703c8369f7",
  "dirty": true
}


## 2. Specification / version

In [2]:
import shaper
from shaper.provenance import file_hash
print("shaper", shaper.__version__)
print("implementation spec hash:", file_hash(shaper.PROTOCOL_SPEC))
print("paper structure hash:  ", file_hash(shaper.PAPER_SPEC))

shaper 1.0.0
implementation spec hash: 635380739cca896e06a3d51c6d00f3b3
paper structure hash:   a1ff800b591eb96ef2fc594caaec897a


## 3. Configuration

In [3]:
DATASET = "ml100k"   # ml1m | beauty | ml100k | synthetic
# ml100k = real MovieLens-100K verification run (not part of the
# registered study); auto-downloads from GroupLens (or a GitHub
# mirror fallback) and exercises the identical staged pipeline.
RUN_ID  = "shaper-run"
cfg = load_run_config(DATASET)
print("config hash:", cfg.config_hash())
print("batch size:", cfg.batch_size, "max_len:", cfg.max_len)
print("seeds:", json.dumps(cfg.seeds, indent=2))

# One gate-aware stage runner: staged-preregistration refusals print a
# readable "GATE:" message instead of a raw SystemExit traceback.
def stage(name, **kw):
    args = run_all.argparse.Namespace(
        dataset=DATASET, run_id=RUN_ID, raw_path=None, force=False,
        skip_validate=False, n_users=48, n_items=24, min_len=6, max_len=9,
        synthetic_seed=7, seeds=None, coalition=None, rec_only_checkpoints="",
        n_permutations=10000, allow_before_archive=False, download=False)
    for k, v in kw.items():
        setattr(args, k, v)
    try:
        return run_all.run_stage(name, cfg, args)
    except SystemExit as exc:
        print(f"GATE: stage '{name}' stopped -> {exc}")
        return exc.code

config hash: 71a3d06011d8dcc791d6f8808d93c958
batch size: 128 max_len: 50
seeds: {
  "recipe_alpha": [
    901,
    902,
    903
  ],
  "pilots": [
    1001,
    1002
  ],
  "engineering": 1001,
  "confirmatory_game_a": [
    2001
  ],
  "extension_game_a": [],
  "game_b": [
    2001
  ],
  "cosine_diagnostic": [
    2001
  ],
  "k4_beauty_mc": [
    3001
  ],
  "nondeterminism_floor_repeat_seed": 2001
}


## 4. Compute estimate (planning only — the paper reports measured values)

In [4]:
def estimate_cost():
    from shaper.cost import n_coalition_models_for_scope
    return n_coalition_models_for_scope(cfg).summary()

print(json.dumps(estimate_cost(), indent=2, default=str))

{
  "items": [
    {
      "label": "recipe_calibration_step1_lr",
      "coalition_models": 3,
      "steps_per_model": 5000,
      "total_steps": 15000,
      "note": "empty coalition, seed 901, 3 LRs"
    },
    {
      "label": "recipe_calibration_step2_curve",
      "coalition_models": 18,
      "steps_per_model": 10000,
      "total_steps": 180000,
      "note": "seeds 901-903 x {2500,5000,10000} x empty+grand (checkpoints reused)"
    },
    {
      "label": "recipe_calibration_step3_lambda_tau",
      "coalition_models": 13,
      "steps_per_model": 0,
      "total_steps": 0,
      "note": "seed 901 (9) + advance 2 pairs to 902,903"
    },
    {
      "label": "pilot_game_a",
      "coalition_models": 16,
      "steps_per_model": 0,
      "total_steps": 0,
      "note": "excluded from confirmatory estimates"
    },
    {
      "label": "primary_game_a",
      "coalition_models": 8,
      "steps_per_model": 0,
      "total_steps": 0,
      "note": "exact K=3, five confirmatory s

## 5. Data

Builds the frozen artifact (maps, splits, roles, quartiles, hashes). For
`ml1m`/`beauty`/`ml100k` the raw archive is downloaded automatically from its
canonical source (`shaper/data_download.py`: streaming download, checksum
verification, mirror fallback, sidecar provenance manifest). `ml100k` is a
verification dataset (reduced model/budget in `configs/ml100k.yaml`; NOT part
of the registered study) whose `u.data` is checked against the canonical
content fingerprint. Archives stay under `data/raw/` (gitignored, never
committed) per the licensing terms.


In [5]:
stage("data", download=True)  # auto-download from the canonical source (checksum-verified)


STAGE data | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/build_data.py --dataset ml100k --run-id shaper-run --download


ml-100k.zip: 100%|██████████| 4.70M/4.70M [00:01<00:00, 3.38MB/s]


[01:47:37] DATA_BUILD           | raw_download                 | info | dataset=ml100k | url=https://files.grouplens.org/datasets/movielens/ml-100k.zip | size=4924029 | sha256=50d2a982c6698693 | verification=published checksum
[01:47:37] DATA_BUILD           | core_iteration               | info | iteration=1 | users=942 | items=1447 | interactions=55375
[01:47:37] DATA_BUILD           | artifact_frozen              | info | dataset=ml100k | data_hash=f4262c74130a2e056ad922f572ba2975 | config_hash=71a3d06011d8dcc791d6f8808d93c958 | users=938 | items=1008 | interactions=54413 | run_id_tag=data-build
{
  "config_hash": "71a3d06011d8dcc791d6f8808d93c958",
  "data_hash": "f4262c74130a2e056ad922f572ba2975",
  "stats": {
    "applicability_rates": {
      "crop": 1.0,
      "mask": 1.0,
      "reorder": 1.0
    },
    "density": 0.05754920127254882,
    "interactions": 54413,
    "items": 1008,
    "positive_before_core": 55375,
    "quartile_edges": [
      17,
      37,
      79
    ],
   

0

## 6. Preflight

In [6]:
from scripts.preflight import run_preflight
print(json.dumps(run_preflight(cfg), indent=2, default=str))

{
  "determinism": {
    "deterministic_mode": true,
    "exception": null,
    "detail": "all kernels deterministic"
  },
  "torch_version": "2.13.0",
  "cuda_available": false,
  "python": "3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]",
  "config_hash": "71a3d06011d8dcc791d6f8808d93c958",
  "batch_size": 128,
  "max_len": 50,
  "status": "PASS",
  "issues": []
}
{
  "determinism": {
    "deterministic_mode": true,
    "exception": null,
    "detail": "all kernels deterministic"
  },
  "torch_version": "2.13.0",
  "cuda_available": false,
  "python": "3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 21.0.0 (clang-2100.0.123.102)]",
  "config_hash": "71a3d06011d8dcc791d6f8808d93c958",
  "batch_size": 128,
  "max_len": 50,
  "status": "PASS",
  "issues": []
}


## 7. Recipe calibration (single pass, V_tune only)

In [7]:
# Selects learning rate, step count, lambda_cl and tau on V_tune and writes
# results/runs/<RUN_ID>/recipe/calibration.json — required by ARCHIVE_FREEZE.
stage("recipe")


STAGE recipe | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/train_recipe.py --dataset ml100k --run-id shaper-run
[01:47:43] RECIPE_CALIBRATION   | stage_start                  | started | dataset=ml100k
[01:47:44] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=901 | coalition= | policy=game_a | steps=12
[01:47:45] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=901 | coalition= | policy=game_a | step=10 | loss=1.36182 | m_c=0 | grad_norm=0.2424
[01:47:45] PRIMARY_GAME_A       | coalition_training_end       | info | dataset=ml100k | seed=901 | coalition= | policy=game_a | wall_seconds=2.46
[01:47:46] PRIMARY_GAME_A       | stale_cache_rejected         | warning | seed=901 | coalition= | policy=game_a | reason=recipe_hash mismatch: 'f50cf2645b28327a6f1261aef1485d9a' != '73d1be412abea2eceb1405250729b28a'
[01:47:46] PRIMARY_GAME_A       | coa

0

## 8. Pilots (excluded seeds 1001/1002)

In [8]:
# The two excluded full-game pilot seeds. The archive cannot be frozen
# without them (the archive is pilot-informed, spec B.7).
stage("pilot")


STAGE pilot | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/train_coalitions.py --dataset ml100k --run-id shaper-run --stage pilot
[01:48:09] PILOT_1001_1002      | stage_start                  | started | dataset=ml100k
[01:48:09] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=1001 | coalition= | policy=game_a | steps=16
[01:48:10] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=1001 | coalition= | policy=game_a | step=3 | loss=1.38602 | m_c=0 | grad_norm=0.2428
[01:48:10] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=1001 | coalition= | policy=game_a | step=6 | loss=1.3761 | m_c=0 | grad_norm=0.2285
[01:48:11] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=1001 | coalition= | policy=game_a | step=9 | loss=1.35773 | m_c=0 | grad_norm=0.2496
[01:48:11] PRIMARY_GAME_A 

0

## 9. Amendment / archive freeze (no confirmatory model before this)

In [9]:
# Order matters: amendment (after pilots), then the archive freeze. The
# archive gate re-checks that the recipe calibration AND the pilot tables of
# THIS run exist, so Section 7 and Section 8 must have completed above.
stage("amendment")
stage("archive")
print("archive frozen:", run_all.archive_is_frozen(cfg))


STAGE amendment | dataset=ml100k run=shaper-run
PILOT_AMENDMENT recorded at /Users/mlouhichi/Desktop/Shapper/results/runs/amendment-ml100k.json
STAGE amendment -> OK in 0.0s

STAGE archive | dataset=ml100k run=shaper-run
ARCHIVE_FREEZE: /Users/mlouhichi/Desktop/Shapper/results/runs/freeze-synthetic-ml100k.yaml frozen (dataset=ml100k)
STAGE archive -> OK in 0.0s
archive frozen: True


## 10. Game A (exact K=3, all 8 coalitions x 5 confirmatory seeds)

In [10]:
stage("game-a")


STAGE game-a | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/train_coalitions.py --dataset ml100k --run-id shaper-run --stage game-a
[01:49:02] PRIMARY_GAME_A       | stage_start                  | started | dataset=ml100k
[01:49:03] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=2001 | coalition= | policy=game_a | steps=16
[01:49:03] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition= | policy=game_a | step=3 | loss=1.39096 | m_c=0 | grad_norm=0.2493
[01:49:04] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition= | policy=game_a | step=6 | loss=1.3712 | m_c=0 | grad_norm=0.2345
[01:49:04] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition= | policy=game_a | step=9 | loss=1.35209 | m_c=0 | grad_norm=0.2278
[01:49:04] PRIMARY_GAME_

0

## 11. Game B (policy sensitivity, seeds 2001–2003, 6 intermediates only)

In [11]:
stage("game-b")


STAGE game-b | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/train_coalitions.py --dataset ml100k --run-id shaper-run --stage game-b
[01:49:30] SECONDARY_GAME_B     | stage_start                  | started | dataset=ml100k
[01:49:30] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=2001 | coalition=crop | policy=game_b | steps=16
[01:49:31] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=crop | policy=game_b | step=3 | loss=1.57796 | m_c=0.333333 | grad_norm=0.4333
[01:49:31] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=crop | policy=game_b | step=6 | loss=1.55797 | m_c=0.333333 | grad_norm=0.3458
[01:49:32] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=crop | policy=game_b | step=9 | loss=1.53731 | m_c=0.333333 | gra

0

## 12. NLL/cosine severity diagnostics (frozen rec-only checkpoints)

In [12]:
# point --rec-only-checkpoints at the recipe empty-coalition checkpoints
print("run: scripts/run_game.py --stage severity --rec-only-checkpoints <paths>")

run: scripts/run_game.py --stage severity --rec-only-checkpoints <paths>


## 13. K=4 Beauty MC (antithetic permutation, <=10 unique models/seed)

In [13]:
stage("k4-mc")


STAGE k4-mc | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/train_coalitions.py --dataset ml100k --run-id shaper-run --stage k4-mc
[01:49:51] K4_BEAUTY_MC         | stage_start                  | started | dataset=ml100k
[01:49:52] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=3001 | coalition= | policy=game_a | steps=16
[01:49:52] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=3001 | coalition= | policy=game_a | step=3 | loss=1.38627 | m_c=0 | grad_norm=0.2276
[01:49:53] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=3001 | coalition= | policy=game_a | step=6 | loss=1.36912 | m_c=0 | grad_norm=0.2669
[01:49:53] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=3001 | coalition= | policy=game_a | step=9 | loss=1.33778 | m_c=0 | grad_norm=0.3002
[01:49:53] PRIMARY_GAME_A

0

## 14. Shapley (exact allocation + efficiency residuals + per-user decomposition)

In [14]:
stage("shapley")


STAGE shapley | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_game.py --dataset ml100k --run-id shaper-run --stage shapley
[01:50:30] SHAPLEY              | stage_start                  | started | dataset=ml100k
[01:50:30] SHAPLEY              | stage_end                    | completed | dataset=ml100k
{
  "game_a": {
    "coalition_spread": [
      {
        "max": 3.01506370306015e-05,
        "min": -0.002449694089591503,
        "range": 0.0024798447266221046,
        "sd": 0.0009090977349984604
      }
    ],
    "contextual_marginals_per_seed": [
      {
        "crop|empty": -0.0001574680209159851,
        "crop|mask": 0.00174686498939991,
        "crop|mask+reorder": 0.0013422174379229546,
        "crop|reorder": 0.0011620130389928818,
        "mask|crop": 0.0001876186579465866,
        "mask|crop+reorder": 0.00026109255850315094,
        "mask|empty": -0.0017167143523693085,
        "mask|reorde

0

## 15. LOO / interactions

In [15]:
stage("loo")
stage("interactions")


STAGE loo | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_game.py --dataset ml100k --run-id shaper-run --stage loo
[01:50:31] LOO                  | stage_start                  | started | dataset=ml100k
[01:50:31] LOO                  | stage_end                    | completed | dataset=ml100k
{
  "2001": {
    "contextual_marginals": {
      "crop|empty": -0.0001574680209159851,
      "crop|mask": 0.00174686498939991,
      "crop|mask+reorder": 0.0013422174379229546,
      "crop|reorder": 0.0011620130389928818,
      "mask|crop": 0.0001876186579465866,
      "mask|crop+reorder": 0.00026109255850315094,
      "mask|empty": -0.0017167143523693085,
      "mask|reorder": 8.088815957307816e-05,
      "reorder|crop": -0.0011302130296826363,
      "reorder|crop+mask": -0.001056739129126072,
      "reorder|empty": -0.002449694089591503,
      "reorder|mask": -0.0006520915776491165
    },
    "grand_loo": {
   

0

## 16. RQ3 — behavioural segments (exact Game A only)

In [16]:
stage("segments")


STAGE segments | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_segments.py --dataset ml100k --run-id shaper-run
[01:50:33] SEGMENTS             | stage_start                  | started | dataset=ml100k
[01:50:35] SEGMENTS             | stage_end                    | completed | dataset=ml100k
{
  "mask_trend_T": -0.0007683606012849949,
  "mask_trend_p": 0.5445,
  "mu": {
    "1": {
      "crop": 0.0019599281497797837,
      "mask": 0.0012264532661058856,
      "reorder": -0.004123781660307712
    },
    "2": {
      "crop": -0.005047094338767368,
      "mask": -0.0007544996124240591,
      "reorder": -0.002316157275632373
    },
    "3": {
      "crop": 0.002189219571855243,
      "mask": -0.0040452010104327545,
      "reorder": -0.0015633912100210374
    },
    "4": {
      "crop": 0.004349742301474217,
      "mask": 0.0018111133312521208,
      "reorder": 0.0021724777006069964
    }
  },
  "n_permutatio

0

## 17. Weight calibration (alpha path on V_select)

In [17]:
stage("weight")


STAGE weight | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_interventions.py --dataset ml100k --run-id shaper-run --stage weight
[01:50:36] WEIGHT_CALIBRATION   | stage_start                  | started | dataset=ml100k
[01:50:37] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=901 | coalition=weighted-b4f386b0 | policy=weighted | steps=16
[01:50:38] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=901 | coalition=weighted-b4f386b0 | policy=weighted | step=3 | loss=1.92213 | m_c=1 | grad_norm=1.0525
[01:50:38] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=901 | coalition=weighted-b4f386b0 | policy=weighted | step=6 | loss=1.85959 | m_c=1 | grad_norm=0.9132
[01:50:39] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=901 | coalition=weighted-b4f386b0 | policy=weighted |

0

## 18. Select calibration (locked activation/no-action rule)

In [18]:
stage("select")


STAGE select | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_interventions.py --dataset ml100k --run-id shaper-run --stage select
[01:51:13] SELECT_CALIBRATION   | stage_start                  | started | dataset=ml100k
{
  "activation": {
    "activated": false,
    "checks": {
      "all_positive_views_80pct_stable": {
        "detail": "unstable views: []",
        "pass": true
      },
      "grand_uplift_ci_above_zero": {
        "detail": "CI95 = (-0.001027, -0.001027)",
        "pass": false
      },
      "highest_weight_sign_stability": {
        "detail": "crop positive in 1/1 seeds (need >= 4)",
        "pass": false
      }
    },
    "deployment_fallback": "rec-only",
    "note": "activation never happens because results look promising",
    "status": "NOT_ACTIVATED"
  },
  "decision": {
    "action": "none",
    "deployment": "rec-only",
    "note": "activation rule failed; rec-only deployme

0

## 19. Baseline controls (LOO, gates, direct search, Dirichlet, random)

In [19]:
stage("controls")


STAGE controls | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_interventions.py --dataset ml100k --run-id shaper-run --stage controls
[01:51:14] BASELINE_CONTROLS    | stage_start                  | started | dataset=ml100k
[01:51:14] PRIMARY_GAME_A       | cache_hit                    | info | dataset=ml100k | seed=901 | coalition=weighted-b4f386b0 | policy=weighted | checkpoint=/Users/mlouhichi/Desktop/Shapper/results/runs/shaper-run/checkpoints/seed901/weighted/weighted-b4f386b0/final.pt
[01:51:15] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=901 | coalition=weighted-d27343e1 | policy=weighted | steps=16
[01:51:16] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=901 | coalition=weighted-d27343e1 | policy=weighted | step=3 | loss=1.94233 | m_c=1 | grad_norm=1.0828
[01:51:16] PRIMARY_GAME_A       | training_step                | inf

0

## 20. Final intervention test (locked decisions; all eligible users; Tables 7A/7B)

In [20]:
stage("final-test")


STAGE final-test | dataset=ml100k run=shaper-run
+ /Users/mlouhichi/Desktop/Shapper/.venv/bin/python /Users/mlouhichi/Desktop/Shapper/scripts/run_interventions.py --dataset ml100k --run-id shaper-run --stage final-test
[01:53:00] FINAL_INTERVENTION_TEST | stage_start                  | started | dataset=ml100k
[01:53:01] PRIMARY_GAME_A       | coalition_training_start     | info | dataset=ml100k | seed=2001 | coalition=weighted-077a72e6 | policy=weighted | steps=16
[01:53:02] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=weighted-077a72e6 | policy=weighted | step=3 | loss=1.94854 | m_c=1 | grad_norm=1.0785
[01:53:03] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=weighted-077a72e6 | policy=weighted | step=6 | loss=1.92847 | m_c=1 | grad_norm=0.7241
[01:53:03] PRIMARY_GAME_A       | training_step                | info | dataset=ml100k | seed=2001 | coalition=weighted-077a72e6 | po

0

## 21. Statistics (seed-level CIs, Holm families, descriptive user analyses)

In [21]:
from shaper.stats import seed_summary, holm_adjust, paired_seed_effects
print("seed-level paired effects and Holm adjustment are computed by "
      "shaper.stats; confirmatory unit = seeds")

seed-level paired effects and Holm adjustment are computed by shaper.stats; confirmatory unit = seeds


## 22. Tables

In [22]:
from shaper.report import shapley_table, coalition_value_table
print("tables are rendered to results/runs/<run_id>/tables/ by the stages above")

tables are rendered to results/runs/<run_id>/tables/ by the stages above


## 23. Figures

In [23]:
print("figures are rendered to results/runs/<run_id>/figures/ by the stages above "
      "(Agg backend, headless-safe)")

figures are rendered to results/runs/<run_id>/figures/ by the stages above (Agg backend, headless-safe)


## 24. Compliance

In [24]:
print("compliance: scripts/validate_run.py generates spec_compliance.json/.md; "
      "run the test suite with: python scripts/run_all.py --validate")

compliance: scripts/validate_run.py generates spec_compliance.json/.md; run the test suite with: python scripts/run_all.py --validate


## 25. Reproducibility

In [25]:
from shaper.artifacts import RunDirectory
run = RunDirectory(RUN_ID, results_root=cfg.paths["results"]).create(cfg, resume=True)
run.write_reproducibility_report(cfg)
print("reproducibility report:", run.root + "/reproducibility_report.md")

reproducibility report: /Users/mlouhichi/Desktop/Shapper/results/runs/shaper-run/reproducibility_report.md


## Notebook helpers (status / validate / estimate-cost / resume)

The helpers below only call package/script functions. The notebook is resume-aware:
every stage is idempotent, so re-running any cell continues from completed work.


In [26]:
def status():
    import json
    p = os.path.join(cfg.paths["results"], RUN_ID, "manifest.json")
    if os.path.exists(p):
        return json.load(open(p))["stages"]
    return {"note": "run not started"}

def validate():
    import subprocess
    return subprocess.call([sys.executable, "-m", "pytest", "tests", "-q"],
                           cwd=REPO_ROOT)

def estimate_cost():
    from shaper.cost import n_coalition_models_for_scope
    return n_coalition_models_for_scope(cfg).summary()

def resume():
    for s in run_all.STAGE_ORDER:
        rc = stage(s)
        if rc not in (0, None):
            print("resume stopped at", s)
            return rc
    return 0

print(status())

{'ARCHIVE_FREEZE': {'at': '2026-08-15T01:49:01Z', 'freeze_file': '/Users/mlouhichi/Desktop/Shapper/results/runs/freeze-synthetic-ml100k.yaml', 'status': 'completed'}, 'BASELINE_CONTROLS': {'at': '2026-08-15T01:52:59Z', 'status': 'completed', 'wall_seconds': 104.64}, 'FINAL_INTERVENTION_TEST': {'at': '2026-08-15T01:53:17Z', 'status': 'completed', 'wall_seconds': 16.63}, 'INTERACTIONS': {'at': '2026-08-15T01:50:33Z', 'status': 'completed', 'wall_seconds': 0.0}, 'K4_BEAUTY_MC': {'at': '2026-08-15T01:50:28Z', 'label': 'Monte-Carlo approximate', 'max_unique_models_per_seed': 10, 'policy': 'game_a', 'seeds': [3001], 'status': 'completed', 'tables': {'3001': '/Users/mlouhichi/Desktop/Shapper/results/runs/shaper-run/coalition_tables/k4_seed3001.json'}, 'wall_seconds': 36.94}, 'LOO': {'at': '2026-08-15T01:50:31Z', 'status': 'completed', 'wall_seconds': 0.0}, 'PILOT_1001_1002': {'at': '2026-08-15T01:49:01Z', 'note': 'pilot outcomes excluded from confirmatory estimates', 'policy': 'game_a', 'seed